In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('daily_health.csv')

In [6]:
df.shape

(8499, 11)

In [7]:

df['date'] = pd.to_datetime(df['datetime']).dt.date

## Creating Hourly Squences data

In [8]:
# List of feature columns
feature_cols = ['StepTotal', 'total_sleep_min', 'hourly_heart_rate', 'Calories']
label_cols = ['lack_of_exercise', 'poor_sleep', 'high_heart_rate', 'high_calorie']

In [9]:
X_list = []
y_list = []

In [10]:

for user_id, group in df.groupby('Id'):
    for date, day_group in group.groupby('date'):
        if len(day_group) == 24:  # only take full days
            X_seq = day_group[feature_cols].values  # shape (24, 4)
            y_seq = day_group[label_cols].iloc[0].values  # daily label (same for each row)
            X_list.append(X_seq)
            y_list.append(y_seq)

In [11]:

X = np.array(X_list)
y = np.array(y_list)

In [12]:
print("LSTM Input shape (hourly):", X.shape)   
print("Labels shape:", y.shape)                


LSTM Input shape (hourly): (164, 24, 4)
Labels shape: (164, 4)


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## Creating LSTM Architecture

In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional

def build_simple_lstm(input_shape, output_dim):
    model = Sequential([
        LSTM(32, input_shape=input_shape, return_sequences=False),
        Dense(output_dim, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

def build_deep_lstm(input_shape, output_dim):
    model = Sequential([
        LSTM(64, input_shape=input_shape, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dense(output_dim, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

def build_bi_lstm(input_shape, output_dim):
    model = Sequential([
        Bidirectional(LSTM(32, return_sequences=False), input_shape=input_shape),
        Dense(output_dim, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model


## Multi-Label Stratified Train Test Split

In [13]:
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold


mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)


for train_idx, test_idx in mskf.split(X, y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    break



label_cols = ['lack_of_exercise', 'poor_sleep', 'high_heart_rate', 'high_calorie']

print("Train label counts:")
print(pd.DataFrame(y_train, columns=label_cols).sum())
print("\nTest label counts:")
print(pd.DataFrame(y_test, columns=label_cols).sum())


Train label counts:
lack_of_exercise     18
poor_sleep            9
high_heart_rate      17
high_calorie        114
dtype: int64

Test label counts:
lack_of_exercise     5
poor_sleep           2
high_heart_rate      4
high_calorie        29
dtype: int64


## Train, Predict, and Evaluate - LSTM 1



In [15]:
daily_cals = df.groupby(['Id', 'date'])['Calories'].sum()
print("Daily Calories stats:")
print(daily_cals.describe())
print("Days with calories > 2500:", (daily_cals > 2500).sum())


Daily Calories stats:
count     469.000000
mean     2141.204691
std       826.299035
min        90.000000
25%      1704.000000
50%      2087.000000
75%      2580.000000
max      4551.000000
Name: Calories, dtype: float64
Days with calories > 2500: 124


In [15]:
!pip install iterative-stratification


In [16]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, hamming_loss, classification_report, f1_score



input_shape = (X_train.shape[1], X_train.shape[2])
output_dim = y_train.shape[1]

model_simple = build_simple_lstm(input_shape, output_dim)

print("\nTraining Simple LSTM ...")
model_simple.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1, verbose=0)

y_pred_proba = model_simple.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

print("\n===== LSTM 1 Results =====")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=label_cols))


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Training Simple LSTM ...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step

===== LSTM 1 Results =====
Accuracy: 0.6060606060606061
Classification Report:
                   precision    recall  f1-score   support

lack_of_exercise       0.00      0.00      0.00         4
      poor_sleep       0.00      0.00      0.00         3
 high_heart_rate       0.00      0.00      0.00         0
    high_calorie       1.00      0.40      0.57        10

       micro avg       1.00      0.24      0.38        17
       macro avg       0.25      0.10      0.14        17
    weighted avg       0.59      0.24      0.34        17
     samples avg       0.12      0.12      0.12        17



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _

## Train, Predict, and Evaluate - LSTM 2

In [17]:


input_shape = (X_train.shape[1], X_train.shape[2])
output_dim = y_train.shape[1]

model_deep = build_deep_lstm(input_shape, output_dim)

print("\nTraining Deep LSTM ...")
model_deep.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1, verbose=0)

y_pred_proba = model_deep.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

print("\n===== LSTM 2 Results =====")
print("Subset Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=label_cols))


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Training Deep LSTM ...
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step

===== LSTM 2 Results =====
Subset Accuracy: 0.8787878787878788
Classification Report:
                   precision    recall  f1-score   support

lack_of_exercise       1.00      1.00      1.00         4
      poor_sleep       0.00      0.00      0.00         3
 high_heart_rate       0.00      0.00      0.00         0
    high_calorie       0.90      0.90      0.90        10

       micro avg       0.93      0.76      0.84        17
       macro avg       0.47      0.47      0.47        17
    weighted avg       0.76      0.76      0.76        17
     samples avg       0.39      0.39      0.39        17



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _

## Train, Predict, and Evaluate - LSTM 3

In [18]:

input_shape = (X_train.shape[1], X_train.shape[2])
output_dim = y_train.shape[1]

model_bi = build_bi_lstm(input_shape, output_dim)

print("\nTraining Bidirectional LSTM ...")
model_bi.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.1, verbose=0)

y_pred_proba = model_bi.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

print("\n===== LSTM 3 Results =====")
print("Subset Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred, target_names=label_cols))


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/keras/src/layers/rnn/bidirectional.py:110: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Training Bidirectional LSTM ...
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/stepWARNING:tensorflow:6 out of the last 6 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x16283f1a0> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step

===== LSTM 3 Results =====
Subset Accuracy: 0.7575757575757576
Classification Report:
                   precision    recall  f1-score   support

lack_of_exercise       1

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _

In [19]:
model_deep.save('best_deep_lstm_model.h5')